# 09 · Why do the TCGA lung models transfer to CPTAC lung? (plan 7h, post hoc)

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 30 Oct 2026 · **Needs:** `results/01_inventory/` to `results/08_transfer/` · **Plan:** section
**7h** (approved 30 Oct 2026 after the notebook 08 results were seen, before this notebook was written). Every result
here is **post hoc**; nothing in notebooks 01–08 changes.

**What this notebook does, in plain words**

In notebook 08, models trained on TCGA lung (adenocarcinoma vs squamous) with the patient removed from the scan
("background only") still predicted the class in CPTAC lung, but not in NSCLC Radiogenomics. The background-only scan
has no anatomy, but it still shows the empty body outline, the table, the field of view and the noise texture. This
notebook asks which of these carries the transfer:
1. **the body outline:** does the silhouette (body shape only) transfer too?
2. **the recorded scanner settings, or patient size:** do a model on the DICOM-header fields, or on body size alone,
   transfer?
3. **which fields differ by class** in each of the three lung datasets (descriptive)?
4. **what the transferred score follows** inside each CPTAC class (descriptive)?

Nothing is downloaded and no GPU is needed: only the embeddings and header records saved by notebooks 05–08.

**Rehearsal first.** With the default `RUN_MODE = "both"`, everything first runs on **shuffled labels**, checks itself,
and only then runs on the real labels.

**How to run:** in Colab, any runtime is enough (CPU is fine; more CPUs are faster), then *Runtime → Run all*. Sign in
with **the Google account that holds the project Drive folder** when Drive asks. Expect about 10–30 minutes. Outputs go to
`results/09_transfer_posthoc/`. The last cell stops the runtime.

In [1]:
# 1. Setup: Drive and folders
import os, re, sys, json, time, hashlib, datetime, platform, warnings
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')
assert RUN_MODE in ('both', 'rehearsal', 'real'), RUN_MODE

IN01 = PROJECT / 'results' / '01_inventory'
IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
IN04 = PROJECT / 'results' / '04_posthoc'
IN05 = PROJECT / 'results' / '05_image_check'
IN06 = PROJECT / 'results' / '06_image_posthoc'
IN07 = PROJECT / 'results' / '07_screen_validation'
IN08 = PROJECT / 'results' / '08_transfer'
CACHE05, CACHE06, CACHE07, CACHE08 = (d / 'cache' for d in (IN05, IN06, IN07, IN08))     # read only
OUT_ROOT = PROJECT / 'results' / '09_transfer_posthoc'
for d in (IN01, IN02, IN03, IN04):
    assert (d / 'run_manifest.json').exists(), f'{d} is missing: run the earlier notebooks first.'
for d in (IN05, IN06, IN07, IN08):
    for m in ('rehearsal', 'real'):
        assert (d / m / 'run_manifest.json').exists(), f'{d / m} is missing: run notebooks 05 to 08 first.'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Project folder:', PROJECT)
print('Outputs to    :', OUT_ROOT)
print('Run mode      :', RUN_MODE)
print('Run started   :', RUN_STARTED, '(UTC)')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/09_transfer_posthoc
Run mode      : both
Run started   : 2026-10-30T14:38:52+00:00 (UTC)


In [2]:
# 2. Packages (the analysis packages of notebook 05, checked against its run record below; no image packages needed)
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.stats import rankdata, spearmanr, kruskal, chi2_contingency
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1
N_CPU = os.cpu_count()

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, default=_jsonable))
    tmp.replace(path)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__,
            'cpus': N_CPU}
print(VERSIONS)

{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'joblib': '1.6.0', 'cpus': 2}


## Settings (plan sections 7d, 7g and 7h)
Every model setting is notebook 05's and every transfer setting notebook 08's (both checked against their run records
below). New here: the fields of checks 3 and 4.

In [3]:
# 3. Settings
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part',
               'hdr_kernel', 'hdr_filter_type', 'hdr_patient_position'}
LOG_NUMERIC = {'image_count', 'body_area_cm2'}
TEXT = {'protocol_text', 'care_text'}
HDR_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_pixel_spacing', 'hdr_recon_diameter', 'hdr_kvp',
               'hdr_tube_current', 'hdr_exposure', 'hdr_ctdivol', 'hdr_pitch']
HDR_CATEGORICAL = ['hdr_kernel', 'hdr_filter_type', 'hdr_patient_position']
HDR_COLS = HDR_NUMERIC + HDR_CATEGORICAL
BODY_COLS = ['body_area_cm2']
ENCODERS = ('ctfm', 'merlin')
VARIANTS = ('original', 'background', 'silhouette')
INPUT_NAME = {'original': 'image', 'background': 'background', 'silhouette': 'silhouette'}
GATE_NULL_RANGE = (0.35, 0.55)
REPRO_TOL = 1e-3

# The datasets (notebook 08's names and class order; class 1 is squamous)
SOURCE = 'lung histology (LUAD vs LUSC)'
SOURCES = [SOURCE]
NSCLC_TASK = 'NSCLC Radiogenomics | histology'
TARGET_NAMES = ['CPTAC lung', 'NSCLC Radiogenomics']      # CPTAC: the question of 7h; NSCLC Radiogenomics: contrast
PAIR_OF = {'CPTAC lung': 'lung: TCGA -> CPTAC', 'NSCLC Radiogenomics': 'lung: TCGA -> NSCLC Radiogenomics'}
CLASSES = {SOURCE: ('TCGA-LUAD', 'TCGA-LUSC'), 'NSCLC Radiogenomics': ('adenocarcinoma', 'squamous'),
           'CPTAC lung': ('CPTAC-LUAD', 'CPTAC-LSCC')}
METADATA_TARGETS = ()                                     # notebook 02's features are not needed here

# Check 3 (descriptive): fields compared by class in each dataset; check 4 uses the fields of 7f check 3(c)
FIELDS_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_recon_diameter', 'hdr_kvp', 'hdr_tube_current',
                  'body_area_cm2']
FIELDS_CATEGORICAL = ['manufacturer', 'hdr_kernel', 'hdr_patient_position', 'hdr_contrast_agent_given']
CORR_NUMERIC = HDR_NUMERIC + ['hdr_coverage_mm', 'body_area_cm2']
CORR_CATEGORICAL = HDR_CATEGORICAL + ['hdr_manufacturer']

# Local code tests only (ignored in Colab)
TEST = {}
if not IN_COLAB:
    TEST = {'real_is_shuffled': True}   # local tests never see the real labels
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [ ]:
# 4. Load and verify the inputs from notebooks 01-08; the three lung datasets
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder} changed since that run: {bad}'
    return man

def cache_md5(folder):
    files = sorted((q for q in folder.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv')),
                   key=lambda q: q.relative_to(folder).parts)
    return len(files), hashlib.md5(''.join(md5(q) for q in files).encode()).hexdigest()

man01, man02, man03, man04 = verify(IN01), verify(IN02), verify(IN03), verify(IN04)
man = {nb: {m: verify(d / m) for m in ('rehearsal', 'real')} for nb, d in (('05', IN05), ('06', IN06), ('07', IN07),
                                                                          ('08', IN08))}
runs = {'01': man01['run_started_utc'], '02': man02['run_started_utc'], '03': man03['run_started_utc'],
        '04': man04['run_started_utc']}
for nb in ('05', '06', '07', '08'):
    for m in ('rehearsal', 'real'):
        assert man[nb][m]['input_runs'] == runs, f'notebook {nb} ({m}) was run on other inputs'
        if not TEST:
            assert man[nb][m]['in_colab'] and not man[nb][m]['local_test_settings'], f'notebook {nb} {m} is not the Colab run'
    runs = {**runs, nb: man[nb]['real']['run_started_utc']}
s05, s08 = man['05']['real']['settings'], man['08']['real']['settings']
same = {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'MIN_COUNT': MIN_COUNT, 'LR_PARAMS': LR_PARAMS,
        'HDR_COLS': HDR_COLS}
if not TEST:
    same.update({'N_BOOT': N_BOOT, 'N_PERM': N_PERM})
diff = [k for k, v in same.items() if json.loads(json.dumps(s05[k])) != json.loads(json.dumps(v))]
assert not diff, f'settings differ from notebook 05: {diff}'
assert list(s08['INPUTS_B']) == ['original', 'background'], 'notebook 08 analysed other inputs'
assert all(man['08']['real']['classes'][k] == list(v) for k, v in CLASSES.items()), 'class order differs from notebook 08'
caches = {}
for nb, folder in (('05', CACHE05), ('06', CACHE06), ('07', CACHE07), ('08', CACHE08)):
    n, h = cache_md5(folder)
    assert (n, h) == (man[nb]['real']['cache_files']['count'], man[nb]['real']['cache_files']['combined_md5']), \
        f'notebook {nb} cache differs from its run record'
    caches[nb] = {'count': n, 'combined_md5': h}
v05 = man['05']['real']['versions']
same_versions = (v05['numpy'], v05['pandas'], v05['scipy'], v05['scikit-learn']) == \
                (np.__version__, pd.__version__, scipy.__version__, sklearn.__version__)
print('Inputs verified: notebooks 01-08 and the caches of notebooks 05-08 '
      f"({', '.join(f'{k}: {v['count']} files' for k, v in caches.items())})")
print('Package versions', 'match notebook 05' if same_versions else
      f"DIFFER from notebook 05 (then numpy {v05['numpy']}, pandas {v05['pandas']}, scipy {v05['scipy']}, "
      f"scikit-learn {v05['scikit-learn']}): small differences in the fitted models are possible")

labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
main = pd.read_csv(IN01 / 'main_series.csv', low_memory=False)
KEY = ['collection', 'PatientID', 'modality']
text = (toks0.assign(group=np.where(toks0.category == 'care pathway', 'care_text', 'protocol_text'))
             .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat0.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
POLISH_WORDS = set(man04['word_lists']['POLISH_WORDS'])
polish = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)

# CPTAC lung exactly as notebook 08 built it (every patient with a CT main series; the class is the collection)
cptac = main[main.collection.isin(CLASSES['CPTAC lung']) & (main.modality == 'CT')].copy()
cptac['label'] = cptac.collection
cptac['key'] = cptac.collection + '__' + cptac.PatientID
TARGETS = {'CPTAC lung': cptac.sort_values('key').reset_index(drop=True)}

inv = {nb: pd.read_csv(folder / 'image_inventory.csv') for nb, folder in (('05', CACHE05), ('07', CACHE07),
                                                                            ('08', CACHE08))}
IN05_KEYS, IN07_KEYS, IN08_KEYS = (set(inv[nb][inv[nb].embedded].key) for nb in ('05', '07', '08'))

def emb_file(key, variant):
    """Where a scan's embedding is: notebook 05 (original, background) or 06 (silhouette), notebook 07, or notebook 08."""
    if key in IN05_KEYS:
        return (CACHE06 if variant == 'silhouette' else CACHE05) / 'embeddings' / f'{key}.npz'
    return (CACHE07 if key in IN07_KEYS else CACHE08) / 'embeddings' / f'{key}.npz'

def header_of(key):
    folder = CACHE05 if key in IN05_KEYS else CACHE07 if key in IN07_KEYS else CACHE08
    return json.loads((folder / 'headers' / f'{key}.json').read_text())

def has_all(key):
    for v in VARIANTS:
        p = emb_file(key, v)
        if not p.exists():
            return False
        with np.load(p) as z:
            if not all(f'{e}__{v}' in z.files for e in ENCODERS):
                return False
    return True

lung_keys = set(labels[labels.task_id.isin([SOURCE, NSCLC_TASK])].pipe(lambda d: d.collection + '__' + d.PatientID)) \
            | set(cptac.key)
complete = {k for k in lung_keys if has_all(k)}
print(f'Lung patients with all three inputs: {len(complete)} of {len(lung_keys)}')

Inputs verified: notebooks 01-08 and the caches of notebooks 05-08 (05: 1053 files, 06: 1052 files, 07: 1450 files, 08: 477 files)
Package versions match notebook 05
Lung patients with all three inputs: 376 of 376


In [ ]:
# 5. Model building blocks (identical to notebooks 03 and 04), plus the same models on embeddings (notebook 05, unchanged)
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    return [list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y))
            for r in range(N_REPEATS)]

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, PA, PB):
    """Repeat-averaged AUC(A) - AUC(B) on the same patients, with a paired bootstrap 95% CI."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(np.mean([auc(y[idx], PA[r][idx]) - auc(y[idx], PB[r][idx]) for r in range(PA.shape[0])]))
    return mean_auc(y, PA) - mean_auc(y, PB), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_mean(x):
    rng = np.random.default_rng(SEED)
    x = np.asarray(x, float)
    vals = [x[rng.choice(len(x), len(x), replace=True)].mean() for _ in range(N_BOOT)]
    return float(x.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# --- metadata models (notebook 03) ---
def _fit_fold_df(df, y, cols, tr, te, k):
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(df.iloc[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[te])
    return te, proba

def cv_oof_df(df, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_df)(df, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- the same logistic regression on embeddings (standardized inside each training fold) ---
def _fit_fold_matrix(X, y, tr, te, k, extras):
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(X[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(X[te])
    out = {}
    for name, Xe in extras.items():           # the fitted fold model applied to other inputs of the same patients
        pe = np.full((len(te), k), np.nan)
        ok = np.where(~np.isnan(Xe[te]).any(axis=1))[0]
        if len(ok):
            tmp = np.zeros((len(ok), k))
            tmp[:, m.classes_] = m.predict_proba(Xe[te][ok])
            pe[ok] = tmp
        out[name] = pe
    return te, proba, out

def cv_oof_matrix(X, y, splits, extras=None):
    extras = extras or {}
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_matrix)(X, y, tr, te, k, extras) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    E = {n: np.full((len(splits), len(y), k), np.nan) for n in extras}
    for (r, _, _), (te, proba, out) in zip(jobs, res):
        P[r, te] = proba
        for n, pe in out.items():
            E[n][r, te] = pe
    return P, E

def _fit_fold_combined(df, E, y, cols, tr, te, k):
    ct = column_transformer(cols).fit(df.iloc[tr])
    sc = StandardScaler().fit(E[tr])
    Xtr = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[tr])), sparse.csr_matrix(sc.transform(E[tr]))]).tocsr()
    Xte = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[te])), sparse.csr_matrix(sc.transform(E[te]))]).tocsr()
    m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(Xte)
    return te, proba

def cv_oof_combined(df, E, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_combined)(df, E, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- exact permutation tests on one fixed label-independent split (notebook 03, 7b.6) ---
def encoded_folds_df(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def scaled_folds(X):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(X):
        sc = StandardScaler().fit(X[tr])
        out.append((tr, te, sc.transform(X[tr]), sc.transform(X[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(folds, y):
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

# --- within-group AUC and its exact test (notebook 04, part D) ---
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(folds, y, strata, keep):
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

In [ ]:
# 6. Datasets: notebook 05's task frame and embedding loader, notebook 07's input loader, notebook 08's dataset frame
# (all unchanged), and the header records of every scan
def task_frame(tid, shuffle_seed=None):
    """Task patients with metadata, groups, headers and body size. shuffle_seed: shuffled labels (rehearsal)."""
    df = labels[labels.task_id == tid].merge(feat, on=KEY, how='left', validate='one_to_one')
    df['key'] = df.collection + '__' + df.PatientID
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if shuffle_seed is not None:
        y = np.random.default_rng(shuffle_seed).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(polish, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

def load_embeddings(keys, variants, path_of):
    out = {(e, v): [] for e in ENCODERS for v in variants}
    for k in keys:
        p = path_of(k)
        ok = p is not None and p.exists()
        z = np.load(p) if ok else None
        for e in ENCODERS:
            for v in variants:
                name = f'{e}__{v}'
                out[(e, v)].append(z[name] if ok and name in z.files else None)
        if z is not None:
            z.close()
    res = {}
    for (e, v), lst in out.items():
        d = next((len(a) for a in lst if a is not None), 1)
        res[(e, v)] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in lst]).astype(float)
    return res

def load_inputs(keys):
    """{(encoder, variant): matrix} from whichever notebook holds each scan's embedding."""
    out = {}
    for v in VARIANTS:
        out.update(load_embeddings(keys, (v,), lambda k, v=v: emb_file(k, v)))
    return out

def dataset_frame(name, shuffle_seed=None):
    """Patients with embeddings, labels in the shared class order (class 1: squamous or papillary), metadata features.
    shuffle_seed: shuffled labels (rehearsal). The sources and NSCLC Radiogenomics use notebook 05's task frame, so their
    rows and shuffles are exactly those of notebooks 05 and 07."""
    if name in SOURCES or name == 'NSCLC Radiogenomics':
        df, y, classes = task_frame(name if name in SOURCES else NSCLC_TASK, shuffle_seed)
        assert tuple(classes) == CLASSES[name], f'{name}: class order {classes} differs from {CLASSES[name]}'
    else:
        df = TARGETS[name]
        if name in METADATA_TARGETS:      # notebook 02's features, as notebook 05's task frame adds them
            df = df.drop(columns=[c for c in feat.columns if c not in KEY and c in df]).merge(
                feat, on=KEY, how='left', validate='one_to_one')
        y = (df.label == CLASSES[name][1]).astype(int).to_numpy()
        if shuffle_seed is not None:
            y = np.random.default_rng(shuffle_seed).permutation(y)
    keep = df.key.isin(complete).to_numpy()
    return df[keep].reset_index(drop=True), y[keep], int((~keep).sum())

def with_headers(df):
    """The scan's header record and body size, prepared as in notebooks 05 and 06 (categorical fields as text)."""
    h = pd.DataFrame([header_of(k) for k in df.key])
    wanted = ['body_area_cm2', 'hdr_contrast_agent_given', 'hdr_coverage_mm', 'hdr_manufacturer'] + HDR_COLS
    df = df.drop(columns=[c for c in wanted if c in df]).merge(h[['key'] + [c for c in wanted if c in h]], on='key',
                                                                how='left')
    for c in wanted:
        if c not in df:
            df[c] = np.nan
    for c in HDR_CATEGORICAL + ['hdr_manufacturer']:
        df[c] = df[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
    df['manufacturer'] = df.hdr_manufacturer.str.upper().str.extract(r'^\s*([A-Z]+)', expand=False)   # as experiment C
    return df

In [ ]:
# 7. The checks of plan 7h
def fit_apply(Xa, ya, Xb):
    """The probe (scaling + logistic regression, notebook 05's settings) fitted on all of A, applied to B. The score is
    the log-odds of class 1: the same ranking (and AUC) as the probability, but it cannot round to exactly 0 or 1 when
    B's embeddings lie far from A's (settled item 6)."""
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(Xa, ya)
    assert list(m.classes_) == [0, 1]
    return m.decision_function(Xb)

def fit_apply_df(dfa, ya, dfb, cols):
    """Notebook 03's metadata model fitted on all of A, applied to B (log-odds of class 1, as above)."""
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(dfa, ya)
    assert list(m.classes_) == [0, 1]
    return m.decision_function(dfb)

def as_P(s):
    """One score vector in the (repeats, patients, classes) shape of notebook 05's AUC and bootstrap functions
    (only the class-1 column is read, by rank)."""
    return np.stack([-s, s], axis=1)[None]

def _external_chunk(Xa, ys, Xb, yb):
    return [auc_binary(yb, fit_apply(Xa, yp, Xb)) for yp in ys]

def external_test(Xa, ya, Xb, yb):
    """External AUC; permutation test by refitting on permuted A labels (settled item 6: descriptive, and the gate)."""
    s = fit_apply(Xa, ya, Xb)
    observed = auc_binary(yb, s)
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(ya) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_external_chunk)(Xa, c, Xb, yb) for c in chunks))
    return s, observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), float(null.mean())

def _external_chunk_df(dfa, ys, dfb, yb, cols):
    return [auc_binary(yb, fit_apply_df(dfa, yp, dfb, cols)) for yp in ys]

def external_test_df(dfa, ya, dfb, yb, cols):
    """notebook 08's external test, for a table model (header fields or body size)."""
    s = fit_apply_df(dfa, ya, dfb, cols)
    observed = auc_binary(yb, s)
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(ya) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_external_chunk_df)(dfa, c, dfb, yb, cols) for c in chunks))
    return s, observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), float(null.mean())

def transfers(lo):
    """Notebook 08's rule (7g, settled item 15): a model transfers if its external CI lies wholly above 0.5."""
    return bool(lo > 0.5)

def reading_silhouette(bg_transferred, sil_lo, diff_lo):
    """Plan 7h, check 1, fixed before this notebook was written."""
    if not bg_transferred:
        return 'not applicable (background did not transfer in notebook 08)'
    if transfers(sil_lo):
        return 'may reflect the body outline or scan geometry'
    if diff_lo > 0:
        return 'beyond the body outline'
    return 'unresolved'

def reading_tables(bg_transferred_any, hdr_lo, body_lo):
    """Plan 7h, check 2, fixed before this notebook was written."""
    if not bg_transferred_any:
        return 'not applicable (background did not transfer in notebook 08)'
    parts = (['consistent with recorded acquisition settings'] if transfers(hdr_lo) else []) + \
            (['consistent with patient size'] if transfers(body_lo) else [])
    return ' and '.join(parts) if parts else 'not explained by the recorded header fields or body size'

def boot_auc(y, x):
    """AUC of one field for class 1 (rank-based), with a percentile bootstrap CI over patients; missing values left out."""
    ok = np.isfinite(x)
    y, x = y[ok], x[ok]
    if len(np.unique(y)) < 2 or np.unique(x).size < 2:
        return np.nan, np.nan, np.nan, int(ok.sum())
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) == 2:
            vals.append(auc_binary(y[idx], x[idx]))
    return auc_binary(y, x), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)), int(ok.sum())

def recorded(table, mode, key, col):
    """A number as recorded by an earlier notebook for the same labels (local tests: other shuffles, so none)."""
    if mode == 'real' and TEST.get('real_is_shuffled'):
        return np.nan
    t = table[mode].set_index(list(key.keys()))
    return float(t.loc[tuple(key.values()), col])

def analyse(mode, shuffle_seed):
    t08 = {m: pd.read_csv(IN08 / m / 'transfer.csv') for m in ('rehearsal', 'real')}
    t06s = {m: pd.read_csv(IN06 / m / 'silhouette_probe.csv') for m in ('rehearsal', 'real')}
    t06h = {m: pd.read_csv(IN06 / m / 'background_vs_header.csv') for m in ('rehearsal', 'real')}
    dfs, ys, ex_s = dataset_frame(SOURCE, shuffle_seed)
    dfs = with_headers(dfs)
    es = load_inputs(dfs.key)
    src = {'n_source': len(ys), 'source_classes': ' | '.join(f'{c}: {int((ys == j).sum())}'
                                                              for j, c in enumerate(CLASSES[SOURCE]))}
    # the source's own (internal) AUCs, each checked against the notebook that first computed it
    splits = repeated_splits(ys)
    P_sil = {enc: cv_oof_matrix(es[(enc, 'silhouette')], ys, splits)[0] for enc in ENCODERS}
    P_hdr, P_body = cv_oof_df(dfs, ys, HDR_COLS, splits), cv_oof_df(dfs, ys, BODY_COLS, splits)
    internal = {'header': mean_auc(ys, P_hdr), 'body': mean_auc(ys, P_body),
                **{f'silhouette_{enc}': mean_auc(ys, P_sil[enc]) for enc in ENCODERS}}
    ref = {'header': recorded(t06h, mode, {'task_id': SOURCE, 'encoder': 'ctfm'}, 'header_only_auc'),
           'body': recorded(t06s, mode, {'task_id': SOURCE, 'encoder': 'ctfm'}, 'body_size_auc_nb05'),
           **{f'silhouette_{enc}': recorded(t06s, mode, {'task_id': SOURCE, 'encoder': enc}, 'silhouette_auc')
              for enc in ENCODERS}}
    rec = {'silhouette': [], 'tables': [], 'missing': [], 'numeric': [], 'categorical': [], 'corr': [], 'scores': [],
           'repro': [{'quantity': f'source internal AUC, {k}', 'value': internal[k], 'recorded': ref[k],
                      'reproduced': (bool(abs(internal[k] - ref[k]) < REPRO_TOL) if np.isfinite(ref[k]) else None)}
                     for k in internal]}
    frames = {SOURCE: (dfs, ys)}
    for tgt in TARGET_NAMES:
        dft, yt, ex_t = dataset_frame(tgt, shuffle_seed)
        dft = with_headers(dft)
        frames[tgt] = (dft, yt)
        et = load_inputs(dft.key)
        pair = PAIR_OF[tgt]
        base = {'pair': pair, 'target': tgt, **src, 'n_target': len(yt), 'n_target_excluded_no_image': ex_t,
                'target_classes': ' | '.join(f'{c}: {int((yt == j).sum())}' for j, c in enumerate(CLASSES[tgt]))}
        bg_transferred = {}
        for enc in ENCODERS:
            # check 1: the silhouette's transfer, beside the background's (recomputed; must reproduce notebook 08)
            s_bg = fit_apply(es[(enc, 'background')], ys, et[(enc, 'background')])
            a_bg = auc_binary(yt, s_bg)
            r08 = t08[mode].set_index(['pair', 'encoder', 'input']).loc[(pair, enc, 'background')]
            bg_transferred[enc] = r08.reading != 'does not transfer'
            ref_bg = recorded(t08, mode, {'pair': pair, 'encoder': enc, 'input': 'background'}, 'external_auc')
            rec['repro'].append({'quantity': f'{pair}, {enc}, background external AUC', 'value': a_bg, 'recorded': ref_bg,
                                 'reproduced': (bool(abs(a_bg - ref_bg) < REPRO_TOL) if np.isfinite(ref_bg) else None)})
            s_sil, a_sil, p_sil, null_sil = external_test(es[(enc, 'silhouette')], ys, et[(enc, 'silhouette')], yt)
            lo, hi = boot_ci(yt, as_P(s_sil))
            d, dlo, dhi = boot_diff(yt, as_P(s_bg), as_P(s_sil))
            rec['silhouette'].append({**base, 'encoder': enc, 'background_external_auc': a_bg,
                                      'background_reading_nb08': r08.reading,
                                      'source_internal_silhouette_auc': internal[f'silhouette_{enc}'],
                                      'silhouette_external_auc': a_sil, 'silhouette_ci_low': lo, 'silhouette_ci_high': hi,
                                      'silhouette_perm_p': p_sil, 'silhouette_null_mean_auc': null_sil,
                                      'silhouette_transfers': transfers(lo), 'background_minus_silhouette': d,
                                      'difference_ci_low': dlo, 'difference_ci_high': dhi,
                                      'reading': reading_silhouette(bg_transferred[enc], lo, dlo),
                                      'role': 'question of 7h' if tgt == 'CPTAC lung' else 'contrast (descriptive)'})
            for name, s in (('background', s_bg), ('silhouette', s_sil)):
                rec['scores'].append(pd.DataFrame({'pair': pair, 'model': f'{enc} {name}', 'key': dft.key, 'label': yt,
                                                   'score': s}))
            # check 4: what the transferred background score follows, inside each CPTAC class (as 7f check 3(c))
            if tgt == 'CPTAC lung':
                for j, c in enumerate(CLASSES[tgt]):
                    m = yt == j
                    for f in CORR_NUMERIC:
                        v = dft[f].to_numpy(float)[m]
                        ok = np.isfinite(v)
                        r_ = {'encoder': enc, 'class': c, 'field': f, 'test': 'Spearman rho', 'n': int(ok.sum())}
                        if ok.sum() >= 10 and np.unique(v[ok]).size > 1:
                            s_ = spearmanr(s_bg[m][ok], v[ok])
                            r_.update({'statistic': float(s_.statistic), 'p': float(s_.pvalue)})
                        rec['corr'].append(r_)
                    for f in CORR_CATEGORICAL:
                        g = dft[f][m].to_numpy()
                        levels = [l for l in pd.Series(g).dropna().unique() if (g == l).sum() >= MIN_COUNT]
                        r_ = {'encoder': enc, 'class': c, 'field': f, 'test': 'Kruskal-Wallis H',
                              'n': int(sum((g == l).sum() for l in levels)), 'n_levels': len(levels)}
                        if len(levels) >= 2:
                            try:
                                s_ = kruskal(*[s_bg[m][g == l] for l in levels])
                                r_.update({'statistic': float(s_.statistic), 'p': float(s_.pvalue)})
                            except ValueError:      # all scores identical
                                pass
                        rec['corr'].append(r_)
        # check 2: a model on the header fields, and one on body size alone
        row = {**base}
        for name, cols, Pi in (('header', HDR_COLS, P_hdr), ('body_size', BODY_COLS, P_body)):
            s, a, p, null = external_test_df(dfs, ys, dft, yt, cols)
            lo, hi = boot_ci(yt, as_P(s))
            row.update({f'source_internal_{name}_auc': mean_auc(ys, Pi), f'{name}_external_auc': a,
                        f'{name}_ci_low': lo, f'{name}_ci_high': hi, f'{name}_perm_p': p,
                        f'{name}_null_mean_auc': null, f'{name}_transfers': transfers(lo)})
            rec['scores'].append(pd.DataFrame({'pair': pair, 'model': name, 'key': dft.key, 'label': yt, 'score': s}))
        row['background_transferred_nb08'] = ' | '.join(f'{e}: {bool(v)}' for e, v in bg_transferred.items())
        row['reading'] = reading_tables(any(bg_transferred.values()), row['header_ci_low'], row['body_size_ci_low'])
        row['role'] = 'question of 7h' if tgt == 'CPTAC lung' else 'contrast (descriptive)'
        rec['tables'].append(row)
    # the missing share of every header field, per dataset (the imputer's indicators could act as dataset labels)
    for name, (df, y) in frames.items():
        rec['missing'] += [{'dataset': name, 'field': c, 'n': len(df), 'missing_share': float(df[c].isna().mean())}
                           for c in HDR_COLS + BODY_COLS]
        # check 3: which fields differ by class (class 1: squamous), descriptive
        for f in FIELDS_NUMERIC:
            a, lo, hi, n = boot_auc(y, df[f].to_numpy(float))
            rec['numeric'].append({'dataset': name, 'field': f, 'n': n, 'auc_squamous_higher': a, 'ci_low': lo,
                                   'ci_high': hi, 'median_class0': float(np.nanmedian(df[f].to_numpy(float)[y == 0])),
                                   'median_class1': float(np.nanmedian(df[f].to_numpy(float)[y == 1]))})
        for f in FIELDS_CATEGORICAL:
            g = df[f].astype(object).where(df[f].notna(), 'missing').astype(str)
            ct = pd.crosstab(g, y).reindex(columns=[0, 1], fill_value=0)
            p = float(chi2_contingency(ct.to_numpy())[1]) if ct.shape[0] > 1 else np.nan
            for value, r in ct.iterrows():
                rec['categorical'].append({'dataset': name, 'field': f, 'value': value, 'n_class0': int(r[0]),
                                           'n_class1': int(r[1]), 'share_class0': r[0] / max(ct[0].sum(), 1),
                                           'share_class1': r[1] / max(ct[1].sum(), 1), 'chi2_p': p})
    return rec

In [ ]:
# 8. Run the checks in one mode (rehearsal on shuffled labels, or real), with saving and a run record
def run_mode(mode):
    out = OUT_ROOT / mode
    out.mkdir(parents=True, exist_ok=True)
    shuffle_seed = 999 if mode == 'rehearsal' else (1000 if TEST.get('real_is_shuffled') else None)
    print(f'\n===== {mode.upper()} pass ({"shuffled labels" if shuffle_seed is not None else "REAL labels"}) =====')
    t0 = time.time()
    rec = analyse(mode, shuffle_seed)
    print(f'Checks finished in {(time.time() - t0) / 60:.1f} min')
    tables = {'silhouette_transfer.csv': pd.DataFrame(rec['silhouette']),
              'header_body_transfer.csv': pd.DataFrame(rec['tables']),
              'header_missing_share.csv': pd.DataFrame(rec['missing']),
              'field_differences_numeric.csv': pd.DataFrame(rec['numeric']),
              'field_differences_categorical.csv': pd.DataFrame(rec['categorical']),
              'cptac_background_score_vs_fields.csv': pd.DataFrame(rec['corr']),
              'reproduction_check.csv': pd.DataFrame(rec['repro'])}
    for name, t in tables.items():
        t.to_csv(out / name, index=False)
    pd.concat(rec['scores'], ignore_index=True).to_csv(out / 'transfer_scores.csv.gz', index=False)
    show(tables['silhouette_transfer.csv'][['target', 'encoder', 'background_external_auc', 'silhouette_external_auc',
                                            'silhouette_ci_low', 'silhouette_ci_high', 'background_minus_silhouette',
                                            'difference_ci_low', 'difference_ci_high', 'reading']], 10)
    show(tables['header_body_transfer.csv'][['target', 'header_external_auc', 'header_ci_low', 'header_ci_high',
                                             'body_size_external_auc', 'body_size_ci_low', 'body_size_ci_high',
                                             'reading']], 5)
    outputs = []
    for p in sorted(q for q in out.rglob('*') if q.is_file()):
        if p.name == 'run_manifest.json':
            continue
        n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
        outputs.append({'file': p.relative_to(out).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
    repro = tables['reproduction_check.csv'].reproduced.dropna()
    manifest = {
        'notebook': '09_transfer_posthoc.ipynb', 'mode': mode, 'plan': '7h (post hoc)',
        'labels': ('real' if shuffle_seed is None else 'shuffled (rehearsal)' if mode == 'rehearsal'
                   else 'shuffled (local code test)'), 'run_started_utc': RUN_STARTED,
        'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'in_colab': IN_COLAB, 'local_test_settings': TEST, 'input_runs': runs, 'input_caches': caches,
        'same_versions_as_notebook05': same_versions,
        'reproduced_notebooks06_08': (bool(repro.astype(bool).all()) if len(repro) else None),
        'patients_with_all_inputs': len(complete),
        'settings': {**same, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM, 'REPRO_TOL': REPRO_TOL, 'BODY_COLS': BODY_COLS,
                     'FIELDS_NUMERIC': FIELDS_NUMERIC, 'FIELDS_CATEGORICAL': FIELDS_CATEGORICAL},
        'versions': VERSIONS, 'outputs': outputs,
    }
    (out / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return tables

In [ ]:
# 9. Run: rehearsal (shuffled labels) -> automatic gate -> real labels
modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results:
        r = results['rehearsal']
        nulls = pd.concat([r['silhouette_transfer.csv'].silhouette_null_mean_auc,
                           r['header_body_transfer.csv'].header_null_mean_auc,
                           r['header_body_transfer.csv'].body_size_null_mean_auc])
        problems = []
        if len(r['silhouette_transfer.csv']) != len(TARGET_NAMES) * len(ENCODERS) or \
                len(r['header_body_transfer.csv']) != len(TARGET_NAMES):
            problems.append('not every check finished')
        if not nulls.between(*GATE_NULL_RANGE).all():
            problems.append(f'a null mean AUC is outside {GATE_NULL_RANGE}: {nulls.round(3).tolist()}')
        if len(complete) < 0.95 * len(lung_keys):
            problems.append('too few patients with all three inputs')
        if problems:
            print('\nGATE NOT PASSED; the real-label pass was NOT run:', *problems, sep='\n - ')
            print('Inspect the run log.')
            break
        print('\nRehearsal gate passed: all checks ran, null means look like chance, enough scans.')
        rep = r['reproduction_check.csv'].reproduced.dropna()
        if not rep.astype(bool).all():
            print('WARNING: the rehearsal did not reproduce notebooks 06 and 08 within the margin. The real pass runs, '
                  'but review the run log before interpreting it.')
    results[mode] = run_mode(mode)


===== REHEARSAL pass (shuffled labels) =====
Checks finished in 6.4 min


,target,encoder,background_external_auc,silhouette_external_auc,silhouette_ci_low,silhouette_ci_high,background_minus_silhouette,difference_ci_low,difference_ci_high,reading
0,CPTAC lung,ctfm,0.419670,0.528529,0.393978,0.666704,-0.108859,-0.256019,0.027838,not applicable (background did not transfer in notebook 08)
1,CPTAC lung,merlin,0.440691,0.490240,0.354884,0.628574,-0.049550,-0.237818,0.137203,not applicable (background did not transfer in notebook 08)
2,NSCLC Radiogenomics,ctfm,0.448521,0.500365,0.392763,0.597587,-0.051844,-0.209922,0.108421,not applicable (background did not transfer in notebook 08)
3,NSCLC Radiogenomics,merlin,0.460752,0.377327,0.278395,0.477706,0.083425,-0.059362,0.219426,not applicable (background did not transfer in notebook 08)


,target,header_external_auc,header_ci_low,header_ci_high,body_size_external_auc,body_size_ci_low,body_size_ci_high,reading
0,CPTAC lung,0.504880,0.374238,0.638525,0.549550,0.414629,0.689947,not applicable (background did not transfer in notebook 08)
1,NSCLC Radiogenomics,0.412286,0.302124,0.539990,0.479737,0.385365,0.574255,not applicable (background did not transfer in notebook 08)



Rehearsal gate passed: all checks ran, null means look like chance, enough scans.

===== REAL pass (REAL labels) =====
Checks finished in 6.4 min


,target,encoder,background_external_auc,silhouette_external_auc,silhouette_ci_low,silhouette_ci_high,background_minus_silhouette,difference_ci_low,difference_ci_high,reading
0,CPTAC lung,ctfm,0.677177,0.637387,0.509803,0.760606,0.039790,-0.105565,0.178016,may reflect the body outline or scan geometry
1,CPTAC lung,merlin,0.739489,0.676426,0.536866,0.793364,0.063063,-0.074328,0.203775,may reflect the body outline or scan geometry
2,NSCLC Radiogenomics,ctfm,0.493428,0.458744,0.359279,0.560909,0.034684,-0.098134,0.165539,not applicable (background did not transfer in notebook 08)
3,NSCLC Radiogenomics,merlin,0.438846,0.438664,0.341721,0.533345,0.000183,-0.117702,0.120516,not applicable (background did not transfer in notebook 08)


,target,header_external_auc,header_ci_low,header_ci_high,body_size_external_auc,body_size_ci_low,body_size_ci_high,reading
0,CPTAC lung,0.517267,0.379252,0.656200,0.473724,0.339458,0.611130,not explained by the recorded header fields or body size
1,NSCLC Radiogenomics,0.407722,0.311969,0.506246,0.628697,0.514184,0.731224,not applicable (background did not transfer in notebook 08)


In [ ]:
# 10. Plain-English summary, read back from the saved files
for mode in results:
    out = OUT_ROOT / mode
    sl = pd.read_csv(out / 'silhouette_transfer.csv')
    hb = pd.read_csv(out / 'header_body_transfer.csv')
    rp = pd.read_csv(out / 'reproduction_check.csv')
    lab_kind = json.loads((out / 'run_manifest.json').read_text())['labels']
    print(f"\n{mode.upper()} ({lab_kind}{': every number should look like chance' if lab_kind.startswith('shuffled') else ''})")
    for r in sl.itertuples():
        print(f"- {r.target}, {r.encoder}: background {r.background_external_auc:.3f}, silhouette "
              f"{r.silhouette_external_auc:.3f} ({r.silhouette_ci_low:.3f}-{r.silhouette_ci_high:.3f}); difference "
              f"{r.background_minus_silhouette:.3f} ({r.difference_ci_low:.3f} to {r.difference_ci_high:.3f}) -> {r.reading}")
    for r in hb.itertuples():
        print(f"- {r.target}: header model {r.header_external_auc:.3f} ({r.header_ci_low:.3f}-{r.header_ci_high:.3f}), "
              f"body size {r.body_size_external_auc:.3f} ({r.body_size_ci_low:.3f}-{r.body_size_ci_high:.3f}) -> {r.reading}")
    rr = rp.reproduced.dropna()
    print(f"- reproduction of notebooks 06 and 08: {int(rr.astype(bool).sum())} of {len(rr)} within {REPRO_TOL}")
print('\nNext: review the saved results and run log before writing up findings.')


REHEARSAL (shuffled (rehearsal): every number should look like chance)
- CPTAC lung, ctfm: background 0.420, silhouette 0.529 (0.394-0.667); difference -0.109 (-0.256 to 0.028) -> not applicable (background did not transfer in notebook 08)
- CPTAC lung, merlin: background 0.441, silhouette 0.490 (0.355-0.629); difference -0.050 (-0.238 to 0.137) -> not applicable (background did not transfer in notebook 08)
- NSCLC Radiogenomics, ctfm: background 0.449, silhouette 0.500 (0.393-0.598); difference -0.052 (-0.210 to 0.108) -> not applicable (background did not transfer in notebook 08)
- NSCLC Radiogenomics, merlin: background 0.461, silhouette 0.377 (0.278-0.478); difference 0.083 (-0.059 to 0.219) -> not applicable (background did not transfer in notebook 08)
- CPTAC lung: header model 0.505 (0.374-0.639), body size 0.550 (0.415-0.690) -> not applicable (background did not transfer in notebook 08)
- NSCLC Radiogenomics: header model 0.412 (0.302-0.540), body size 0.480 (0.385-0.574) -> 

In [ ]:
# 11. Stop the runtime once everything above has finished
# "Run all" stops at the first error, so if any cell fails, this cell never runs and the runtime stays on for checking.
if IN_COLAB:
    from google.colab import runtime
    print('Notebook finished:', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))
    drive.flush_and_unmount()   # push every result file to Drive before the machine is released
    print('Drive synced. Stopping the runtime now.')
    time.sleep(5)
    runtime.unassign()